In [2]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 67.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 91.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 88.6 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 50.4 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 88.7 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 89.7 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 90.4 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 97.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 53.4 MB/s  0:00:06m0:00:0100:01
 

In [3]:
# -*- coding: utf-8 -*-
"""
Scoring de pertinence datasets <-> sous-thèmes par embeddings (sentence-transformers).

Entrées :
  - sous_themes_termes : dict {sous-thème: termes descriptifs séparés par des virgules}
    (ex. celui produit par extraire_theme / filtrer_datasets_par_theme)
  - datasets : DataFrame des datasets du thème (colonnes notamment
    'tags', 'title', 'description')
  - une liste de sous-thèmes à scorer (par défaut : tous ceux du dict)

Sorties :
  - datasets_scores : copie du DataFrame d'entrée augmentée d'une colonne
    "score__<sous-thème>" par sous-thème (similarité cosinus, multi-label)
  - datasets_par_sous_theme : dict {sous-thème: DataFrame des datasets dont
    le score dépasse le seuil, triée par score décroissant}

Modèle recommandé : intfloat/multilingual-e5-large (~2,2 Go) ; alternative
plus légère sur CPU : intfloat/multilingual-e5-base (~1,1 Go, 768 dim).
Le script gère le préfixe "query: " / "passage: " requis par les modèles e5.
"""

import re
import unicodedata
from typing import Iterable

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

# --------------------------------------------------------------------------- #
# Paramètres
# --------------------------------------------------------------------------- #

MODELE = "intfloat/multilingual-e5-large"   # ou "intfloat/multilingual-e5-base"
SEUIL_SCORE = 0.50        # similarité cosinus minimale pour "pertinent"
TAILLE_LOT = 64           # datasets embeddés par lot (mémoire CPU maîtrisée)
PREFIXE_QUERY = "query: "     # préfixe e5 pour les textes de référence
PREFIXE_PASSAGE = "passage: "  # préfixe e5 pour les textes des datasets
MAX_CARACT_DESCRIPTION = 1500  # troncature : descriptions data.gouv parfois très longues


# --------------------------------------------------------------------------- #
# Préparation des textes
# --------------------------------------------------------------------------- #

def _nettoyer(texte) -> str:
    """Concaténation propre : remplace les séparateurs par des espaces."""
    if texte is None or (isinstance(texte, float) and pd.isna(texte)):
        return ""
    return re.sub(r"\s+", " ", str(texte)).strip()


def texte_dataset(row: pd.Series) -> str:
    """Texte représentant un dataset : tags répétés (poids lexical) + titre + description tronquée."""
    tags = _nettoyer(row.get("tags")).replace(",", " ")
    titre = _nettoyer(row.get("title"))
    description = _nettoyer(row.get("description"))[:MAX_CARACT_DESCRIPTION]
    return f"{tags} {tags} {titre} {titre} {description}"


def phrases_sous_theme(termes: str) -> list[str]:
    """
    Découpe les termes descriptifs d'un sous-thème en phrases individuelles.

    Chaque terme devient une phrase séparée, dont on prendra le centroïde
    des embeddings : des descripteurs multiples et contrastés séparent
    mieux les sous-thèmes proches qu'un texte unique trop général.
    """
    return [t.strip() for t in str(termes).split(",") if t.strip()]


In [5]:


# --------------------------------------------------------------------------- #
# Embeddings
# --------------------------------------------------------------------------- #

def charger_modele(nom_modele: str = MODELE) -> SentenceTransformer:
    """
    Charge le modèle en CPU. Sur SSP Cloud :
      - pip install -U sentence-transformers
      - le 1er appel télécharge le modèle (~2 Go pour e5-large) puis le met
        en cache ; prévoir un service avec 4-8 Go de RAM.
    """
    return SentenceTransformer(nom_modele, device="cpu")


def embedder_datasets(
    modele: SentenceTransformer,
    datasets: pd.DataFrame,
    taille_lot: int = TAILLE_LOT,
) -> np.ndarray:
    """
    Embeddings des datasets (un vecteur normalisé par dataset).

    Traitement par lots pour limiter la mémoire ; quelques centaines à
    ~4000 datasets = < 1 minute sur CPU avec e5-base, quelques minutes
    avec e5-large.
    """
    textes = [
        PREFIXE_PASSAGE + texte_dataset(row)
        for _, row in datasets.iterrows()
    ]
    # normalize_embeddings=True : vecteurs unitaires -> similarité cosinus = produit scalaire
    return modele.encode(
        textes,
        batch_size=taille_lot,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )


def embedder_sous_theme(
    modele: SentenceTransformer,
    termes: str,
) -> np.ndarray:
    """
    Embedding d'un sous-thème : centroïde des embeddings de chacun de ses
    termes descriptifs (préfixés "query: "), puis renormalisation.
    """
    phrases = phrases_sous_theme(termes)
    if not phrases:
        raise ValueError("Sous-thème sans termes descriptifs.")
    vecteurs = modele.encode(
        [PREFIXE_QUERY + p for p in phrases],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    centroide = vecteurs.mean(axis=0)
    return centroide / (np.linalg.norm(centroide) + 1e-12)


# --------------------------------------------------------------------------- #
# Pipeline complet
# --------------------------------------------------------------------------- #

def scorer_datasets_par_sous_theme(
    sous_themes_termes: dict[str, str],
    datasets: pd.DataFrame,
    sous_themes_a_scoring: Iterable[str] | None = None,
    modele: SentenceTransformer | None = None,
    seuil: float = SEUIL_SCORE,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Calcule, pour chaque sous-thème, la similarité cosinus entre l'embedding
    du sous-thème et celui de chaque dataset.

    Renvoie :
      (datasets_scores, datasets_par_sous_theme)

    - datasets_scores : copie de `datasets` + une colonne "score__<sous-thème>"
    - datasets_par_sous_theme : {sous-thème: datasets avec score >= seuil,
      triés par score décroissant, avec la colonne de score}

    Remarques :
      - le modèle n'est chargé qu'une fois, puis réutilisé pour tous les
        sous-thèmes (les datasets ne sont embeddés qu'une seule fois)
      - la sortie est multi-label : un dataset peut être pertinent pour
        plusieurs sous-thèmes
      - les scores cosinus e5 sont typiquement dans [0.2, 0.75] pour des
        textes liés ; ajuster `seuil` après inspection (0,5 est une
        valeur de départ raisonnable, à étalonner sur un échantillon validé)
    """
    if modele is None:
        modele = charger_modele()

    if sous_themes_a_scoring is None:
        sous_themes_a_scoring = list(sous_themes_termes.keys())

    # 1. Embeddings des datasets (une seule fois)
    if verbose:
        print(f"Embedding de {len(datasets)} datasets...")
    emb_datasets = embedder_datasets(modele, datasets)

    # 2. Pour chaque sous-thème : embedding (centroïde des termes) + similarité
    datasets_scores = datasets.copy()
    datasets_par_sous_theme = {}

    for st in sous_themes_a_scoring:
        if st not in sous_themes_termes:
            raise KeyError(f"Sous-thème '{st}' absent du dict des termes.")
        emb_st = embedder_sous_theme(modele, sous_themes_termes[st])

        # vecteurs normalisés : produit scalaire = similarité cosinus
        scores = emb_datasets @ emb_st
        datasets_scores[f"score__{st}"] = scores

        selection = datasets_scores[datasets_scores[f"score__{st}"] >= seuil]
        datasets_par_sous_theme[st] = selection.sort_values(
            f"score__{st}", ascending=False
        )
        if verbose:
            print(
                f"Sous-thème '{st}' : {len(selection)}/{len(datasets)} datasets "
                f"au-dessus du seuil {seuil:.2f} "
                f"(score max : {scores.max():.3f}, médiane : {np.median(scores):.3f})"
            )

    return datasets_scores, datasets_par_sous_theme




In [ ]:
# --------------------------------------------------------------------------- #
# Exemple d'utilisation
# --------------------------------------------------------------------------- #

if __name__ == "__main__":
    # from filtre_thematique import filtrer_datasets_par_theme
    # sous_themes, datasets_theme = filtrer_datasets_par_theme("Stationnement")
    # datasets_theme = datasets_theme.reset_index(drop=True)

    # Exemple minimal de dict {sous-thème: termes}
    sous_themes = {
        "Stationnement en voirie": (
            "stationnement en voirie, stationnement réglementé, horodateur, "
            "places de stationnement"
        ),
        "Parcs et ouvrages": (
            "parking, parc en ouvrage, parc-relais, parking payant, parking gratuit"
        ),
    }

    # datasets_theme = ...  # DataFrame issu du filtrage par thème
    # model = charger_modele("intfloat/multilingual-e5-large")
    # scores, par_sous_theme = scorer_datasets_par_sous_theme(
    #     sous_themes_termes=sous_themes,
    #     datasets=datasets_theme,
    #     modele=model,
    #     seuil=0.50,
    # )
    #
    # # Vue validation manuelle : top datasets par sous-thème
    # for st, df in par_sous_theme.items():
    #     print(f"\n=== {st} ===")
    #     print(df[["title", f"score__{st}"]].head(10).to_string())

In [6]:
sous_themes={'Stationnement en voirie': 'stationnement en voirie, stationnement réglementé, horodateur, places de stationnement',
 'Parcs et ouvrages': 'parking, parc en ouvrage, parc-relais, parking payant, parking gratuit',
 'Tarification': 'tarifs parking, tarif, abonnement'}

In [7]:
ds_st=pd.read_csv("data/datasets_stationnt.csv")

,Unnamed: 0,id,url,slug,tags,title,description,badges,owner_id,organization,organization_id,created_at,last_modified,resources_count,resources_formats
0,98,6abfd3f530053ff3563eaf78,https://www.data.gouv.fr/datasets/1-suivi-des-...,1-suivi-des-stationnements-de-puffins-des-bale...,inspire,1- Suivi des stationnements de Puffins des Bal...,La donnée localise les sites choisis pour le s...,inspire,NaN,Office Français de la Biodiversité,5d7f937b8b4c4165718fbff1,2026-04-29T00:00:00,2026-10-02T15:55:33.538000,1,gml
1,144,6abfc99d30053ff3563eaf4b,https://www.data.gouv.fr/datasets/2-suivi-des-...,2-suivi-des-stationnements-de-puffins-des-bale...,inspire,2- Suivi des stationnements de Puffins des Bal...,La donnée localise les points d'observation ef...,inspire,NaN,Office Français de la Biodiversité,5d7f937b8b4c4165718fbff1,2026-04-29T00:00:00,2026-10-02T15:11:24.821000,1,gml
2,155,6abfc79f30053ff3563eaf40,https://www.data.gouv.fr/datasets/suivi-des-st...,suivi-des-stationnements-de-puffins-des-balear...,inspire,Suivi des stationnements de Puffins des Baléar...,Dans le cadre du Plan national d'actions (PNA)...,inspire,NaN,Office Français de la Biodiversité,5d7f937b8b4c4165718fbff1,2026-04-17T00:00:00,2026-10-02T15:02:55.133000,0,NaN
3,157,6abfc79730053ff3563eaf3e,https://www.data.gouv.fr/datasets/3-suivi-des-...,3-suivi-des-stationnements-de-puffins-des-bale...,inspire,3- Suivi des stationnements de Puffins des Bal...,La donnée localise les observations réalisées ...,inspire,NaN,Office Français de la Biodiversité,5d7f937b8b4c4165718fbff1,2026-04-29T00:00:00,2026-10-02T15:02:47.257000,1,gml
4,727,6abb3833c98f21f136ff6371,https://www.data.gouv.fr/datasets/plui-station...,plui-stationnements,"amenagement-urbanisme-foncier,document-durbani...",PLUi-Stationnements,Règles de stationnement du PLUi applicables au...,inspire,NaN,Métropole Européenne de Lille,58b69c62c751df4b175dd7a4,2026-09-17T10:35:42,2026-09-29T04:01:55.330000,2,"ogc:wms,ogc api features"


In [9]:
model = charger_modele("intfloat/multilingual-e5-large")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4337.73it/s]


In [10]:
scores, par_sous_theme = scorer_datasets_par_sous_theme(
    sous_themes_termes=sous_themes,
    datasets=ds_st,
    modele=model,
    seuil=0.50,
    )

Embedding de 789 datasets...


Batches: 100%|██████████| 13/13 [05:00<00:00, 23.12s/it]


Sous-thème 'Stationnement en voirie' : 789/789 datasets au-dessus du seuil 0.50 (score max : 0.895, médiane : 0.847)
Sous-thème 'Parcs et ouvrages' : 789/789 datasets au-dessus du seuil 0.50 (score max : 0.877, médiane : 0.833)
Sous-thème 'Tarification' : 789/789 datasets au-dessus du seuil 0.50 (score max : 0.864, médiane : 0.812)


In [ ]:
scores.columns

Index(['Unnamed: 0', 'id', 'url', 'slug', 'tags', 'title', 'description',
       'badges', 'owner_id', 'organization', 'organization_id', 'created_at',
       'last_modified', 'resources_count', 'resources_formats',
       'score__Stationnement en voirie', 'score__Parcs et ouvrages',
       'score__Tarification'],
      dtype='str')

In [12]:
scores.to_csv("data/ds_stationnt_scores.csv")

In [13]:
par_sous_theme

{'Stationnement en voirie':      Unnamed: 0                        id  \
 380       47877  655cc1d3f7d2a99a97ddd078   
 29         2063  6a82d884f2338802736c07dd   
 584       59702  6088a69cb8d03e27059bacda   
 532       56449  614bfca3da9cd74eedb0681a   
 748       69545  56b0c2f4a3a7294d38b88a80   
 ..          ...                       ...   
 366       47144  659d808c331b620d12df109d   
 771       71003  54e0b9e1c751df0a5b467389   
 28         1993  6a8712b1e9c0caa7b2d36732   
 203       22946  67a2d497c89e7afa36c6a493   
 153       20539  67ff1f69e9e446c411f0e026   
 
                                                    url  \
 380  https://www.data.gouv.fr/datasets/stationnemen...   
 29   https://www.data.gouv.fr/datasets/mobilite-hor...   
 584  https://www.data.gouv.fr/datasets/stationnemen...   
 532   https://www.data.gouv.fr/datasets/horodateurs-11   
 748    https://www.data.gouv.fr/datasets/horodateurs-9   
 ..                                                 ...   
 366  